This notebook can be used as a utility to export a SR model to ONNX format using a training checkpoint.
Includes validation to ensure ONNX model does not performance worse than the checkpoint.

In [1]:
from pathlib import Path
import numpy as np
import torch
import onnx
import os

# Import model here
# from training.train_utils.fsrcnn import FSRCNN

Defining the name of the ONNX file about to be created. The checkpoint to be exported in ONNX format must
follow the name scheme `<model_name>_epoch_<epoch no>.pth` OR `<model_name>_best.pth` (e.g.,
`fsrcnn_epoch_20.pth`, `srcnn_best.pth`)

In [2]:
weights_file_name = r""
model_name = r""
project_root = Path(os.getcwd()).parent
weights_file = project_root / "checkpoints" / model_name / weights_file_name
file_suffix = weights_file_name.replace(f"{model_name}_", "").replace(".pth", "")
onnx_file = project_root / "models" / f"{model_name}_{file_suffix}.onnx"
onnx_file.parent.mkdir(parents=True, exist_ok=True)

Project root: U:\Year_III\DeepLearning\Project


Configuration used in the inference run used to create the ONNX model. **MUST match the configuration used
to train the model**.

In [3]:
scale_factor = 4
num_channels = 3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Project root: U:\Year_III\DeepLearning\Project
Checkpoint: U:\Year_III\DeepLearning\Project\checkpoints\fsrcnn\fsrcnn_best.pth
ONNX output: U:\Year_III\DeepLearning\Project\models\fsrcnn_best_NO.onnx
Device: cpu


Instatiate the model and load the weights from the checkpoint `.pth` file.

In [4]:
# Find weights file
if not weights_path.exists():
    raise FileNotFoundError(f"Checkpoint not found: {weights_path}")

# Instantiate model and load weights
model = FSRCNN(scale_factor=scale_factor, num_channels=num_channels).to(device)
checkpoint = torch.load(weights_path, map_location=device)

# Load state dictionary and normalize
if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    checkpoint = checkpoint["state_dict"]
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    checkpoint = checkpoint["model_state_dict"]

checkpoint = {
    key.removeprefix("module."): value
    for key, value in checkpoint.items()
}

missing_keys, unexpected_keys = model.load_state_dict(checkpoint, strict=False)

if missing_keys or unexpected_keys:
    print("Missing keys:", missing_keys)
    print("Unexpected keys:", unexpected_keys)

# Switch to evaluation mode for inference
model.eval()

NameError: name 'FSRCNN' is not defined

Create a dummy input for inference. The input width and height correspond to an image that has been
downscaled by a factor of <scale_factor> from the original 178x218 px resolution (the standard image size
used in the CelebA dataset).

In [ ]:
# create dummy input for exporting, include batch dimension
input_width = 178 // scale_factor
input_height = 218 // scale_factor
dummy_input = torch.randn(1, num_channels, input_height, input_width, device=device)

# run inference
with torch.no_grad():
    torch_output = model(dummy_input)

The `export()` method runs inference on the dummy input and creates a graph based on how the input gets
processed in the forward pass.
- opset (operation set) version is chosen as 17 (not too old, not too new)
- `dynamic_axes` lists the dimensions of the `ls` and `sr` vectors that might change during inference.
Batch might change since multiple images in one batch may be passed to the model, and the height and width
of the input image changes for upscaling.

In [ ]:
# Export to ONNX
opset_version = 17
torch.onnx.export(
    model,
    dummy_input,
    str(onnx_file),
    export_params=True,
    opset_version=opset_version,
    do_constant_folding=True,
    input_names=["lr"],
    output_names=["sr"],
    dynamic_axes={
        "lr": {0: "batch", 2: "height", 3: "width"},
        "sr": {0: "batch", 2: "height_out", 3: "width_out"},
    },
)

print(f"Exported ONNX model to {onnx_file}")
print(f"File size: {onnx_file.stat().st_size / (1024 * 1024):.2f} MB")

In [ ]:
# Load onnx model and check its validity/consistency
onnx_model = onnx.load(str(onnx_file))
onnx.checker.check_model(onnx_model)
print("ONNX checker validation passed.")

Optionally, we can compare the ONNX output with the PyTorch checkpoint output.

In [ ]:
# Get available providers and use CUDA if available, otherwise use CPU
providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if device.type == "cuda" else [
    "CPUExecutionProvider"]
available_providers = ort.get_available_providers()
providers = [provider for provider in providers if provider in available_providers]

# Create inference session
session = ort.InferenceSession(str(onnx_file), providers=providers)
print("ONNX Runtime providers:", session.get_providers())
print("Input:", session.get_inputs()[0].name, session.get_inputs()[0].shape)
print("Output:", session.get_outputs()[0].name, session.get_outputs()[0].shape)

# Run inference
onnx_input = dummy_input.detach().cpu().numpy().astype(np.float32)
onnx_output = session.run(["sr"], {"lr": onnx_input})[0]

# Convert output to numpy and compare with PyTorch output
# (generated before)
torch_output_np = torch_output.detach().cpu().numpy()

max_abs_diff = np.max(np.abs(torch_output_np - onnx_output))
mean_abs_diff = np.mean(np.abs(torch_output_np - onnx_output))

print("ONNX output shape:", onnx_output.shape)
print(f"Max absolute difference vs PyTorch: {max_abs_diff:.8f}")
print(f"Mean absolute difference vs PyTorch: {mean_abs_diff:.8f}")

# Assert difference between ONNX output and checkpoint output
# Should be less than 10^(-3)% and less than 10^(-4).
np.testing.assert_allclose(onnx_output, torch_output_np, rtol=1e-3, atol=1e-4)

print("ONNX Runtime output matches PyTorch output within tolerance.")